In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
def scrape(path, headless=False):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless")
    chrome_options.add_argument("--window-size=1920x1080")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--hide-scrollbars")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.get(path)
    driver.maximize_window()
    time.sleep(1)
    
    # ==============================================================================================
#     # Complete login
    try:
        provided_u_name = "haider1805@icloud.com"
        user_name = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.NAME, "username")))   
        user_name.send_keys(provided_u_name)
        cont = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, "nextButton")))   
        cont.click()
        time.sleep(1)

        provided_pass = "Muhssan7865"
        password = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.NAME, "password")))
        password.send_keys(provided_pass)
        cont2 = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, "loginBtn")))   
        cont2.click()

    except Exception as e:
        print(f"Login failed: {e}")
        
    time.sleep(1)
    
    # ==============================================================================================
    # Handle cookie consent banner
    try:
        cookie_accept = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, './/button[@id="onetrust-accept-btn-handler"]')))
        cookie_accept.click()
    except Exception as e:
        print(f"Cookie banner handling error: {e}")

    try:
        tom = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, './/span[text()="Tomorrow"]')))
        tom.click()
    except:
        pass
    
    result = []  
    try:
        auc_name = WebDriverWait(driver, 5).until(EC.presence_of_all_elements_located((By.XPATH, './/h4[@class="sc-gKAblj gnhPZo"]'))) 
        auc_times = WebDriverWait(driver, 5).until(EC.presence_of_all_elements_located((By.XPATH, './/p[@class="sc-gKAblj cMXcYv SaleInfo___StyledTypography5-xizojj-9 icGnMC"]'))) 
        auc_lots = WebDriverWait(driver, 5).until(EC.presence_of_all_elements_located((By.XPATH, './/p[@class="sc-gKAblj efkMLX SaleInfo___StyledTypography-xizojj-5 fFwCxX"]'))) 
        if len(auc_name) == len(auc_times) == len(auc_lots):
            for auc, time_, lot in zip(auc_name, auc_times, auc_lots):
                
                details = {}
                
                details['Auction name'] = auc.text.strip()
                details['Time']  = time_.text.strip().split(" ")[0]
                details['Lots']  = lot.text.strip().split(" ")[0]
                
                result.append(details)
                
        else:
            details['Auction name'] = 'na'
    except:
        print("no auc name found")

    df = pd.DataFrame.from_dict(result)
    df.to_csv("Auctions.csv", index=False)
    driver.quit()
    
path = "https://www.bca.co.uk/sales"
scrape(path)
